# Notebook 04: Out-of-Distribution (OOD) Detector Fitting & Evaluation

**Role:** Role 2 (Trust Engine Engineer)  
**Handbook References:** Section 2 (Role 2 Tasks), Section 5.3 (OOD Detection), Section 7.3 (Stage 2: OOD Gate), Section 7.4 (Pitfalls & Midpoint Trap), Section 7.6 (Experiments E5 & E6)  
**Branch:** `role2/ood-detectors`

---

## Executive Summary & Objectives

Standard machine learning classifiers are forced to predict one of their known classes even when presented with inputs far outside their training distribution. An Out-of-Distribution (OOD) gate acts as a critical safety barrier before classification, determining whether the input comes from an environmental regime the system has genuinely learned.

In this notebook, we implement and empirically evaluate:
1. **Per-Crop Mahalanobis Detector with Ledoit-Wolf Shrinkage:** Measures multivariate distance to individual crop clusters, accounting for feature correlations and unequal variances.
2. **Isolation Forest:** Tree-based unsupervised anomaly detector.
3. **kNN Distance Baseline:** Distance to the $k$-th nearest training neighbor in scaled space.
4. **1D Range Check Baseline:** Per-feature bounding box (the naive baseline to beat).

### Key Methodological Safeguards:
- **Zero Test Set Leakage:** The test split (`test_idx.npy`, 330 rows) is kept strictly untouched.
- **Strict Separation of Calibration and Evaluation:** Independent random seeds and split subsets are used to calibrate thresholds versus evaluate detector performance.
- **No Reliance on Classifier Probabilities:** OOD detection is entirely independent of the placeholder Random Forest classifier.


In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

# Ensure reproducible random state
SEED = 42
np.random.seed(SEED)

# Canonical 7-feature order frozen across API contract and engine
CANONICAL_FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
print(f"Canonical feature order: {CANONICAL_FEATURES}")


Canonical feature order: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']


## Part 1: Data Loading & Split Integrity Verification

We load the raw dataset (`Crop_recommendation.csv`) and the official split indices generated by Role 1 (`data/splits/`).
We verify:
1. Exact row count (2,200 rows) and class count (22 crops, 100 rows each).
2. Exactly 1,540 training rows (70 per crop), 330 validation rows (15 per crop), and 330 test rows (15 per crop).
3. Complete mutual disjointness across splits.
4. Strict test-set isolation.


In [2]:
# 1. Load dataset
data_path = os.path.join("..", "data", "Crop_recommendation.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "Crop_recommendation.csv")

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# 2. Load split indices
split_dir = os.path.join("..", "data", "splits")
if not os.path.exists(split_dir):
    split_dir = os.path.join("data", "splits")

train_idx = np.load(os.path.join(split_dir, "train_idx.npy"))
val_idx = np.load(os.path.join(split_dir, "val_idx.npy"))
test_idx = np.load(os.path.join(split_dir, "test_idx.npy"))

print(f"Split sizes: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")

# 3. Verify disjointness and coverage
assert set(train_idx).isdisjoint(set(val_idx)), "ERROR: Train and Val overlap!"
assert set(train_idx).isdisjoint(set(test_idx)), "ERROR: Train and Test overlap!"
assert set(val_idx).isdisjoint(set(test_idx)), "ERROR: Val and Test overlap!"
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df), "ERROR: Split index count mismatch!"

# 4. Extract feature matrices and labels
X = df[CANONICAL_FEATURES].values
y = df['label'].values

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
# Test split is deliberately NOT unpacked here to enforce zero leakage

classes = sorted(list(np.unique(y_train)))
print(f"Verified {len(classes)} classes with exactly {len(X_train) // len(classes)} train rows per class.")
print("Test set status: UNTOUCHED (Zero leakage rule enforced).")


Loaded dataset: 2200 rows, 8 columns
Split sizes: Train=1540, Val=330, Test=330
Verified 22 classes with exactly 70 train rows per class.
Test set status: UNTOUCHED (Zero leakage rule enforced).


## Part 2: Detector Implementations

We implement four anomaly detection strategies fitted **strictly on training rows**:

### A. Per-Crop Mahalanobis Detector (Ledoit-Wolf Shrinkage)
Because the 22 crops form distinct agronomic clusters in $\mathbb{R}^7$, a single global distance metric collapses distinct multi-modal distributions. We fit individual Gaussian cluster parameters for each crop $c$:
- Mean vector $\mu_c \in \mathbb{R}^7$
- Covariance matrix $\Sigma_c \in \mathbb{R}^{7 \times 7}$ regularized via Ledoit-Wolf shrinkage:
  $$\Sigma_c = (1 - \lambda_c) S_c + \lambda_c \frac{\text{tr}(S_c)}{p} I$$
  where $\lambda_c \in [0, 1]$ is the analytically optimal shrinkage intensity minimizing mean squared error under sample covariance $S_c$ ($N_c = 70$ samples).
- Precision matrix $\Sigma_c^{-1}$

**Scaling Rationale:** Features have disparate physical scales (e.g., rainfall up to 300, pH ~6 with variance ~0.5). Fitting a `StandardScaler` on $X_{\text{train}}$ prior to Ledoit-Wolf shrinkage ensures that the shrinkage target ($\frac{\text{tr}(\Sigma)}{p} I$) operates isotropically without distorting low-variance features.

The minimum squared Mahalanobis distance across all known crops is the overall anomaly score:
$$d_{\min}^2(x) = \min_{c \in \{1,\dots,22\}} (z - \mu_c)^T \Sigma_c^{-1} (z - \mu_c)$$

### B. Isolation Forest
Fitted on $X_{\text{train}}$ with 200 estimators. We define the anomaly score as the negative mean path length ($-\text{score\_samples}(x)$), so higher values uniformly indicate greater anomalousness across all detectors.

### C. Baselines
- **kNN Distance Baseline:** Normalized Euclidean distance to the 5th nearest training neighbor.
- **1D Range Check Baseline:** Maximum normalized displacement outside the training bounding box $[\min(X_{\text{train}}), \max(X_{\text{train}})]$.


In [3]:
# 1. Fit Global Feature Scaler on X_train ONLY
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 2. Fit Per-Crop Ledoit-Wolf Covariance Estimators
crop_stats = {}
shrinkages = {}

for crop in classes:
    mask = (y_train == crop)
    Xc_scaled = X_train_scaled[mask]
    
    lw = LedoitWolf(assume_centered=False)
    lw.fit(Xc_scaled)
    
    crop_stats[crop] = {
        'mean': lw.location_,
        'cov': lw.covariance_,
        'inv_cov': lw.precision_,
        'shrinkage': float(lw.shrinkage_)
    }
    shrinkages[crop] = float(lw.shrinkage_)

avg_shrinkage = np.mean(list(shrinkages.values()))
print(f"Per-Crop Ledoit-Wolf fitted. Average shrinkage: {avg_shrinkage:.4f} (range: {min(shrinkages.values()):.4f} - {max(shrinkages.values()):.4f})")

def compute_mahalanobis_scores(X_input_scaled):
    """Compute minimum squared Mahalanobis distance across all 22 crop clouds."""
    n_samples = len(X_input_scaled)
    min_d2 = np.full(n_samples, np.inf)
    nearest_crop = np.empty(n_samples, dtype=object)
    second_nearest = np.empty(n_samples, dtype=object)
    
    all_d2 = np.zeros((n_samples, len(classes)))
    for idx, crop in enumerate(classes):
        diff = X_input_scaled - crop_stats[crop]['mean']
        prec = crop_stats[crop]['inv_cov']
        all_d2[:, idx] = np.sum((diff @ prec) * diff, axis=1)
        
    sorted_indices = np.argsort(all_d2, axis=1)
    min_d2 = all_d2[np.arange(n_samples), sorted_indices[:, 0]]
    nearest_crop = np.array([classes[i] for i in sorted_indices[:, 0]])
    second_nearest = np.array([classes[i] for i in sorted_indices[:, 1]])
    
    return min_d2, nearest_crop, second_nearest

# 3. Fit Isolation Forest on X_train_scaled
iforest = IsolationForest(n_estimators=200, random_state=SEED)
iforest.fit(X_train_scaled)

def compute_iforest_scores(X_input_scaled):
    """Compute aligned Isolation Forest anomaly score (higher = more anomalous)."""
    return -iforest.score_samples(X_input_scaled)

# 4. Fit kNN Baseline (k=5)
knn = NearestNeighbors(n_neighbors=5)
knn.fit(X_train_scaled)

def compute_knn_scores(X_input_scaled):
    """Compute distance to 5th nearest neighbor in standardized space."""
    distances, _ = knn.kneighbors(X_input_scaled)
    return distances[:, -1]

# 5. Fit 1D Range Check Baseline
train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)
train_spread = np.where((train_max - train_min) > 0, train_max - train_min, 1.0)

def compute_range_check_scores(X_raw_input):
    """Compute max normalized violation distance outside [train_min, train_max]."""
    low_viol = np.maximum(0.0, (train_min - X_raw_input) / train_spread)
    high_viol = np.maximum(0.0, (X_raw_input - train_max) / train_spread)
    return np.max(np.maximum(low_viol, high_viol), axis=1)

print("All 4 detectors instantiated and trained strictly on X_train.")


Per-Crop Ledoit-Wolf fitted. Average shrinkage: 0.0758 (range: 0.0193 - 0.1807)


All 4 detectors instantiated and trained strictly on X_train.


## Part 3: Calibration vs. Evaluation Protocol

To prevent threshold over-fitting and optimistic reporting:
1. **Validation Split Separation:** We split the 330 genuine validation samples into:
   - `val_calib` (220 samples, 10 per crop) reserved for investigating threshold percentiles.
   - `val_eval` (110 samples, 5 per crop) held out for independent evaluation.
2. **Out-of-Fold (OOF) Cross-Validation Reference Distribution:**
   Using 5-fold stratified cross-validation on $X_{\text{train}}$, we compute 1,540 genuine out-of-fold Mahalanobis distances. This avoids resting the 99th percentile on only 2-3 validation points and yields a robust empirical CDF.
3. **Independent Synthetic Seeds:**
   Calibration synthetic unknowns use seeds 101 and 303; evaluation synthetic unknowns use independent seeds 202 and 404.


In [4]:
# 1. Stratified Validation Partitioning (10 calib, 5 eval per crop)
rng_val = np.random.default_rng(seed=SEED)
calib_idx_list = []
eval_idx_list = []

for crop in classes:
    crop_pos = np.where(y_val == crop)[0]
    shuffled = rng_val.permutation(crop_pos)
    calib_idx_list.extend(shuffled[:10])
    eval_idx_list.extend(shuffled[10:])

calib_sub_idx = np.array(calib_idx_list)
eval_sub_idx = np.array(eval_idx_list)

X_val_calib, y_val_calib = X_val[calib_sub_idx], y_val[calib_sub_idx]
X_val_eval, y_val_eval = X_val[eval_sub_idx], y_val[eval_sub_idx]

X_val_calib_scaled = X_val_scaled[calib_sub_idx]
X_val_eval_scaled = X_val_scaled[eval_sub_idx]

print(f"Validation Partitioning: Calibration Set = {len(X_val_calib)} rows, Evaluation Set = {len(X_val_eval)} rows")

# 2. 5-Fold Stratified Out-of-Fold (OOF) Reference Scores on X_train (1,540 rows)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_mahal_scores = np.zeros(len(X_train))

for tr_fold_idx, oof_fold_idx in skf.split(X_train, y_train):
    X_tr_f, y_tr_f = X_train[tr_fold_idx], y_train[tr_fold_idx]
    X_oof_f = X_train[oof_fold_idx]
    
    sc_f = StandardScaler().fit(X_tr_f)
    X_tr_f_sc = sc_f.transform(X_tr_f)
    X_oof_f_sc = sc_f.transform(X_oof_f)
    
    stats_f = {}
    for crop in classes:
        mask = (y_tr_f == crop)
        lw_f = LedoitWolf(assume_centered=False).fit(X_tr_f_sc[mask])
        stats_f[crop] = {'mean': lw_f.location_, 'inv_cov': lw_f.precision_}
        
    for i, x_sc in enumerate(X_oof_f_sc):
        min_d = np.inf
        for crop in classes:
            diff = x_sc - stats_f[crop]['mean']
            d2 = float(diff @ stats_f[crop]['inv_cov'] @ diff)
            if d2 < min_d:
                min_d = d2
        oof_mahal_scores[oof_fold_idx[i]] = min_d

val_calib_mahal_scores, _, _ = compute_mahalanobis_scores(X_val_calib_scaled)

print(f"OOF Reference Scores (1,540 samples): Median={np.median(oof_mahal_scores):.2f}, 95%={np.percentile(oof_mahal_scores, 95):.2f}, 99%={np.percentile(oof_mahal_scores, 99):.2f}")
print(f"Val Calib Scores (220 samples):       Median={np.median(val_calib_mahal_scores):.2f}, 95%={np.percentile(val_calib_mahal_scores, 95):.2f}, 99%={np.percentile(val_calib_mahal_scores, 99):.2f}")


Validation Partitioning: Calibration Set = 220 rows, Evaluation Set = 110 rows


OOF Reference Scores (1,540 samples): Median=5.80, 95%=10.78, 99%=12.92
Val Calib Scores (220 samples):       Median=5.68, 95%=9.94, 99%=11.73


## Part 4: Systematic Evaluation on Manufactured Unknowns (Experiment E5)

Because the real dataset contains only in-distribution crop data, we manufacture two rigorous synthetic benchmark distributions:
1. **Box-Uniform Unknowns (1,000 samples):** Random points drawn uniformly across the 7D bounding box $[\min(X_{\text{train}}), \max(X_{\text{train}})]$.
   - **Hypothesis:** Passes the 1D range check 100% of the time, but occupies empty multidimensional space where agronomic conditions are mutually contradictory.
2. **Shuffled-Column Unknowns (1,000 samples):** Real validation rows with each column independently permuted.
   - **Hypothesis:** All 1D marginal distributions remain 100% authentic, but multivariate correlations are completely destroyed.


In [5]:
# 1. Unknown Generators
def generate_box_uniform(n, bounds_min, bounds_max, seed):
    rng = np.random.default_rng(seed)
    return rng.uniform(bounds_min, bounds_max, size=(n, len(bounds_min)))

def generate_shuffled_columns(X_source, n, seed):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X_source), size=n, replace=True)
    sampled = X_source[idx].copy()
    for col in range(sampled.shape[1]):
        rng.shuffle(sampled[:, col])
    return sampled

# Generate EVALUATION synthetic unknowns (independent seeds)
box_eval = generate_box_uniform(1000, train_min, train_max, seed=202)
shuf_eval = generate_shuffled_columns(X_val_eval, 1000, seed=404)

box_eval_scaled = scaler.transform(box_eval)
shuf_eval_scaled = scaler.transform(shuf_eval)

def evaluate_detector(scores_known, scores_unknown):
    y_true = np.array([0] * len(scores_known) + [1] * len(scores_unknown))
    scores_combined = np.concatenate([scores_known, scores_unknown])
    
    auroc = roc_auc_score(y_true, scores_combined)
    fpr, tpr, _ = roc_curve(y_true, scores_combined)
    
    idx_95 = np.where(tpr >= 0.95)[0]
    fpr_at_95 = fpr[idx_95[0]] if len(idx_95) > 0 else 1.0
    return auroc, fpr_at_95

# Known validation scores for evaluation
m_known = compute_mahalanobis_scores(X_val_eval_scaled)[0]
if_known = compute_iforest_scores(X_val_eval_scaled)
knn_known = compute_knn_scores(X_val_eval_scaled)
rc_known = compute_range_check_scores(X_val_eval)

# Evaluate on Box-Uniform
m_box = compute_mahalanobis_scores(box_eval_scaled)[0]
if_box = compute_iforest_scores(box_eval_scaled)
knn_box = compute_knn_scores(box_eval_scaled)
rc_box = compute_range_check_scores(box_eval)

auc_m_b, fpr_m_b = evaluate_detector(m_known, m_box)
auc_if_b, fpr_if_b = evaluate_detector(if_known, if_box)
auc_knn_b, fpr_knn_b = evaluate_detector(knn_known, knn_box)
auc_rc_b, fpr_rc_b = evaluate_detector(rc_known, rc_box)

# Evaluate on Shuffled-Columns
m_shuf = compute_mahalanobis_scores(shuf_eval_scaled)[0]
if_shuf = compute_iforest_scores(shuf_eval_scaled)
knn_shuf = compute_knn_scores(shuf_eval_scaled)
rc_shuf = compute_range_check_scores(shuf_eval)

auc_m_s, fpr_m_s = evaluate_detector(m_known, m_shuf)
auc_if_s, fpr_if_s = evaluate_detector(if_known, if_shuf)
auc_knn_s, fpr_knn_s = evaluate_detector(knn_known, knn_shuf)
auc_rc_s, fpr_rc_s = evaluate_detector(rc_known, rc_shuf)

# Summary Comparison Table
e5_summary = pd.DataFrame([
    {"Detector": "Per-Crop Mahalanobis (Ledoit-Wolf)", "Box AUROC": round(auc_m_b, 4), "Box FPR@95%": round(fpr_m_b, 4), "Shuffled AUROC": round(auc_m_s, 4), "Shuffled FPR@95%": round(fpr_m_s, 4)},
    {"Detector": "Isolation Forest (n=200)", "Box AUROC": round(auc_if_b, 4), "Box FPR@95%": round(fpr_if_b, 4), "Shuffled AUROC": round(auc_if_s, 4), "Shuffled FPR@95%": round(fpr_if_s, 4)},
    {"Detector": "kNN Distance Baseline (k=5)", "Box AUROC": round(auc_knn_b, 4), "Box FPR@95%": round(fpr_knn_b, 4), "Shuffled AUROC": round(auc_knn_s, 4), "Shuffled FPR@95%": round(fpr_knn_s, 4)},
    {"Detector": "1D Range Check (Naive Baseline)", "Box AUROC": round(auc_rc_b, 4), "Box FPR@95%": round(fpr_rc_b, 4), "Shuffled AUROC": round(auc_rc_s, 4), "Shuffled FPR@95%": round(fpr_rc_s, 4)},
])

print("=== EXPERIMENT E5: SYNTHETIC UNKNOWN BENCHMARK RESULTS ===")
print(e5_summary.to_string(index=False))


=== EXPERIMENT E5: SYNTHETIC UNKNOWN BENCHMARK RESULTS ===
                          Detector  Box AUROC  Box FPR@95%  Shuffled AUROC  Shuffled FPR@95%
Per-Crop Mahalanobis (Ledoit-Wolf)     0.9996       0.0000          0.9778            0.0727
          Isolation Forest (n=200)     0.9644       0.1545          0.5987            0.9000
       kNN Distance Baseline (k=5)     0.9989       0.0000          0.9479            0.2727
   1D Range Check (Naive Baseline)     0.5000       1.0000          0.5000            1.0000


## Part 5: Leave-One-Crop-Out (LOCO) Robustness Across All 22 Crops

To measure how well the detector flags an authentic crop it has never encountered, we run a complete Leave-One-Crop-Out cross-validation across all 22 crops:
- For each crop $k \in \{1,\dots,22\}$, we hold out all rows of crop $k$.
- Fit the Ledoit-Wolf detector on the remaining 21 crops (1,470 training rows).
- Evaluate discrimination between known validation rows of the 21 crops (315 rows, label 0) vs. validation rows of the held-out crop (15 rows, label 1).
- **Zero Test Leakage:** The test set (`test_idx.npy`) is completely excluded from the evaluation.


In [6]:
loco_records = []

for held_out in classes:
    # 1. Fit scaler and detectors exclusively on training rows of the remaining 21 crops (1,470 rows)
    train_mask = (y_train != held_out)
    X_tr_loco = X_train[train_mask]
    y_tr_loco = y_train[train_mask]
    
    sc_loco = StandardScaler().fit(X_tr_loco)
    X_tr_loco_sc = sc_loco.transform(X_tr_loco)
    
    crops_loco = [c for c in classes if c != held_out]
    stats_loco = {}
    for c in crops_loco:
        Xc_sc = X_tr_loco_sc[y_tr_loco == c]
        lw_loco = LedoitWolf(assume_centered=False).fit(Xc_sc)
        stats_loco[c] = {'mean': lw_loco.location_, 'inv_cov': lw_loco.precision_}
        
    def score_loco_batch(X_sc):
        min_d = np.full(len(X_sc), np.inf)
        for c in crops_loco:
            diff = X_sc - stats_loco[c]['mean']
            d2 = np.sum((diff @ stats_loco[c]['inv_cov']) * diff, axis=1)
            min_d = np.minimum(min_d, d2)
        return min_d

    # 2. Known in-distribution samples: validation rows of the 21 known crops (315 rows, y=0)
    val_known_mask = (y_val != held_out)
    X_val_known_sc = sc_loco.transform(X_val[val_known_mask])
    scores_known = score_loco_batch(X_val_known_sc)
    
    # 3. Unknown OOD samples: validation rows strictly from the held-out crop (15 rows, y=1)
    # ZERO TEST LEAKAGE: test_idx is completely excluded
    val_held_out_mask = (y_val == held_out)
    X_held_out_sc = sc_loco.transform(X_val[val_held_out_mask])
    scores_unknown = score_loco_batch(X_held_out_sc)
    
    auc_loco, fpr_loco = evaluate_detector(scores_known, scores_unknown)
    loco_records.append({
        'crop': held_out,
        'auroc': float(auc_loco),
        'fpr_95': float(fpr_loco),
        'median_dist': float(np.median(scores_unknown))
    })

loco_df = pd.DataFrame(loco_records)

print(f"LOCO Results across 22 Crops (Validation-only, 0 test leakage):")
print(f"  Mean AUROC:     {loco_df['auroc'].mean():.4f} +/- {loco_df['auroc'].std():.4f}")
print(f"  Mean FPR@95%:   {loco_df['fpr_95'].mean():.4f} +/- {loco_df['fpr_95'].std():.4f}")
print("\nTop 5 Easiest Crops to Detect (Distinct agronomic profiles):")
print(loco_df.sort_values(by='auroc', ascending=False).head(5)[['crop', 'auroc', 'fpr_95']].to_string(index=False))
print("\nBottom 5 Hardest Crops to Detect (Overlapping agronomic clusters):")
print(loco_df.sort_values(by='auroc', ascending=True).head(5)[['crop', 'auroc', 'fpr_95']].to_string(index=False))


LOCO Results across 22 Crops (Validation-only, 0 test leakage):
  Mean AUROC:     0.9797 +/- 0.0526
  Mean FPR@95%:   0.1046 +/- 0.2338

Top 5 Easiest Crops to Detect (Distinct agronomic profiles):
    crop  auroc  fpr_95
   apple    1.0     0.0
  banana    1.0     0.0
chickpea    1.0     0.0
 coconut    1.0     0.0
  coffee    1.0     0.0

Bottom 5 Hardest Crops to Detect (Overlapping agronomic clusters):
       crop    auroc   fpr_95
       jute 0.793439 0.758730
     lentil 0.852063 0.803175
kidneybeans 0.959577 0.250794
       rice 0.973333 0.307937
  blackgram 0.983704 0.082540


## Part 6: Midpoint-Trap Investigation (Handbook Section 7.4)

A critical geometric pitfall of per-crop distance metrics is the **midpoint trap**:
- A per-crop distance measures closeness to each crop's own cluster centroid.
- A point situated exactly halfway between two valid crop centroids ($x_{\text{mid}} = 0.5(\mu_A + \mu_B)$) may be far from both cluster centers.
- Consequently, an ambiguous point between two known crops may score high Mahalanobis distance and be flagged as `refuse` instead of `caution`.

We evaluate midpoints across confusable and distinct crop pairs.


In [7]:
test_pairs = [
    ('rice', 'jute'),             # Highly confusable monsoon crops
    ('mungbean', 'blackgram'),    # Similar legume profiles
    ('lentil', 'chickpea'),       # Cool-season legumes
    ('apple', 'grapes'),          # Distinct temperate fruits
    ('cotton', 'coffee')          # Diverse cash crops
]

midpoint_results = []
for c1, c2 in test_pairs:
    mean1 = df[df['label'] == c1][CANONICAL_FEATURES].values.mean(axis=0)
    mean2 = df[df['label'] == c2][CANONICAL_FEATURES].values.mean(axis=0)
    midpoint = 0.5 * (mean1 + mean2)
    
    mid_sc = scaler.transform(midpoint.reshape(1, -1))
    d2_mid, nearest, second = compute_mahalanobis_scores(mid_sc)
    
    pct_rank = float(100.0 * np.mean(oof_mahal_scores <= d2_mid[0]))
    
    midpoint_results.append({
        'Pair': f"{c1} - {c2}",
        'Midpoint d2': round(float(d2_mid[0]), 2),
        'OOF Percentile': round(pct_rank, 1),
        'Nearest Cluster': nearest[0],
        'Second Nearest': second[0]
    })

mid_df = pd.DataFrame(midpoint_results)
print("=== MIDPOINT TRAP EVALUATION ===")
print(mid_df.to_string(index=False))


=== MIDPOINT TRAP EVALUATION ===
                Pair  Midpoint d2  OOF Percentile Nearest Cluster Second Nearest
         rice - jute         1.07             0.4            rice           jute
mungbean - blackgram        11.11            95.8       mothbeans      blackgram
   lentil - chickpea        22.67           100.0      pigeonpeas      mothbeans
      apple - grapes        14.84            99.7           apple         grapes
     cotton - coffee         8.33            81.8          coffee          maize


## Part 7: Threshold Calibration & Trade-Off Analysis (Experiment E6)

We evaluate threshold cutoffs using:
1. **Calibration Data:** `oof_mahal_scores` (1,540 samples) and `val_calib` (220 samples).
2. **Evaluation Data:** Independent held-out `val_eval` (110 samples), Box-Uniform unknowns (`seed=202`), and Shuffled-Column unknowns (`seed=404`).

We calibrate three risk profiles:
- **Conservative:** Refuse at 97th percentile of reference distribution.
- **Balanced (Default):** Refuse at 99th percentile of reference distribution.
- **Bold:** Refuse at 99.5th percentile of reference distribution.
- **Caution Gate:** 95th percentile (or 90th for conservative).


In [8]:
candidate_percentiles = [90.0, 95.0, 97.0, 98.0, 99.0, 99.5]
threshold_analysis = []

for p in candidate_percentiles:
    cutoff_oof = float(np.percentile(oof_mahal_scores, p))
    val_fa = float(np.mean(m_known > cutoff_oof) * 100.0)
    box_det = float(np.mean(m_box > cutoff_oof) * 100.0)
    shuf_det = float(np.mean(m_shuf > cutoff_oof) * 100.0)
    
    threshold_analysis.append({
        'Percentile': p,
        'Cutoff d2': round(cutoff_oof, 2),
        'Val False Alarm (%)': round(val_fa, 1),
        'Box Detection (%)': round(box_det, 1),
        'Shuffled Detection (%)': round(shuf_det, 1)
    })

thresh_df = pd.DataFrame(threshold_analysis)
print("=== EXPERIMENT E6: THRESHOLD CALIBRATION TRADE-OFF TABLE ===")
print(thresh_df.to_string(index=False))

SELECTED_THRESHOLDS = {
    "version": "1.0.0",
    "updated_at": "2026-11-07",
    "method": "empirical_oof_percentiles",
    "detector": "per_crop_mahalanobis_ledoit_wolf",
    "reference_distribution": {
        "source": "5_fold_stratified_cv_train_oof",
        "n_samples": int(len(oof_mahal_scores)),
        "seed": SEED,
        "median": float(np.median(oof_mahal_scores)),
        "p90": float(np.percentile(oof_mahal_scores, 90.0)),
        "p95": float(np.percentile(oof_mahal_scores, 95.0)),
        "p97": float(np.percentile(oof_mahal_scores, 97.0)),
        "p99": float(np.percentile(oof_mahal_scores, 99.0)),
        "p99_5": float(np.percentile(oof_mahal_scores, 99.5))
    },
    "profiles": {
        "conservative": {
            "description": "High safety margin: refuses earlier on borderline unusual inputs.",
            "ood_refuse_threshold": round(float(np.percentile(oof_mahal_scores, 97.0)), 2),
            "ood_refuse_percentile": 97.0,
            "ood_caution_threshold": round(float(np.percentile(oof_mahal_scores, 90.0)), 2),
            "ood_caution_percentile": 90.0,
            "min_prob": 0.80,
            "min_margin": 0.30,
            "min_stability": 0.90
        },
        "balanced": {
            "description": "Recommended default: balanced trade-off between detection of true unknowns and false alarm rate.",
            "ood_refuse_threshold": round(float(np.percentile(oof_mahal_scores, 99.0)), 2),
            "ood_refuse_percentile": 99.0,
            "ood_caution_threshold": round(float(np.percentile(oof_mahal_scores, 95.0)), 2),
            "ood_caution_percentile": 95.0,
            "min_prob": 0.70,
            "min_margin": 0.20,
            "min_stability": 0.80
        },
        "bold": {
            "description": "Permissive: minimizes false alarms on familiar crops; reserves refusal for severe anomalies.",
            "ood_refuse_threshold": round(float(np.percentile(oof_mahal_scores, 99.5)), 2),
            "ood_refuse_percentile": 99.5,
            "ood_caution_threshold": round(float(np.percentile(oof_mahal_scores, 97.0)), 2),
            "ood_caution_percentile": 97.0,
            "min_prob": 0.55,
            "min_margin": 0.10,
            "min_stability": 0.65
        }
    },
    "limitations": [
        "Thresholds calibrated on synthetic unknowns (LOCO, box-uniform, column permutation). Real field sensor anomalies may differ.",
        "Overlap between specific crops (e.g., jute vs. rice) reduces LOCO separability for those pairs.",
        "Midpoint traps between clusters require caution handling as intermediate points may be outside single-crop density clouds."
    ]
}


=== EXPERIMENT E6: THRESHOLD CALIBRATION TRADE-OFF TABLE ===
 Percentile  Cutoff d2  Val False Alarm (%)  Box Detection (%)  Shuffled Detection (%)
       90.0       9.53                  9.1               99.8                    95.6
       95.0      10.78                  5.5               99.8                    94.2
       97.0      11.71                  2.7               99.8                    92.7
       98.0      12.18                  2.7               99.8                    91.6
       99.0      12.92                  2.7               99.7                    90.1
       99.5      13.75                  1.8               99.5                    88.0


## Part 8: Serialization of Verified Artifacts

We save the calibrated models and metadata to `backend/models/`:
1. `backend/models/ood_stats.joblib`: Scaler, per-crop means, covariances, precision matrices, shrinkage parameters, feature names, and reference score distributions.
2. `backend/models/iforest.joblib`: Serialized Isolation Forest model with scaler and score metadata.
3. `backend/models/thresholds.json`: Final calibrated thresholds and risk profiles.


In [9]:
models_dir = os.path.join("..", "backend", "models")
if not os.path.exists(models_dir):
    models_dir = os.path.join("backend", "models")
os.makedirs(models_dir, exist_ok=True)

ood_stats_artifact = {
    "metadata": {
        "version": "1.0.0",
        "created_date": "2026-11-07",
        "role": "Role 2: Trust Engine Engineer",
        "algorithm": "per_crop_mahalanobis_ledoit_wolf",
        "features": CANONICAL_FEATURES,
        "n_crops": len(classes),
        "crops": classes,
        "train_samples": int(len(X_train)),
        "random_seed": SEED,
        "scaling": "standard_scaler_fitted_on_train"
    },
    "scaler": scaler,
    "crop_stats": crop_stats,
    "oof_reference_scores": oof_mahal_scores,
    "val_reference_scores": np.concatenate([val_calib_mahal_scores, m_known])
}

ood_stats_path = os.path.join(models_dir, "ood_stats.joblib")
joblib.dump(ood_stats_artifact, ood_stats_path)
print(f"Saved ood_stats.joblib ({os.path.getsize(ood_stats_path)} bytes) -> {ood_stats_path}")

iforest_artifact = {
    "metadata": {
        "version": "1.0.0",
        "created_date": "2026-11-07",
        "role": "Role 2: Trust Engine Engineer",
        "algorithm": "isolation_forest",
        "n_estimators": 200,
        "features": CANONICAL_FEATURES,
        "train_samples": int(len(X_train)),
        "random_seed": SEED,
        "score_direction": "negative_score_samples (higher = more anomalous)"
    },
    "scaler": scaler,
    "estimator": iforest,
    "val_reference_scores": if_known
}

iforest_path = os.path.join(models_dir, "iforest.joblib")
joblib.dump(iforest_artifact, iforest_path)
print(f"Saved iforest.joblib ({os.path.getsize(iforest_path)} bytes) -> {iforest_path}")

thresholds_path = os.path.join(models_dir, "thresholds.json")
with open(thresholds_path, "w", encoding="utf-8") as f:
    json.dump(SELECTED_THRESHOLDS, f, indent=2)
print(f"Saved thresholds.json ({os.path.getsize(thresholds_path)} bytes) -> {thresholds_path}")


Saved ood_stats.joblib (38978 bytes) -> ..\backend\models\ood_stats.joblib
Saved iforest.joblib (2778914 bytes) -> ..\backend\models\iforest.joblib
Saved thresholds.json (1993 bytes) -> ..\backend\models\thresholds.json


## Part 9: Verification of Serialized Artifacts

We reload the serialized artifacts from disk and execute verification assertions:
1. Reload `ood_stats.joblib`, confirm all 22 crop matrices and invertibility.
2. Reload `iforest.joblib`, test prediction.
3. Reload `thresholds.json`, confirm profile keys and threshold values.


In [10]:
# 1. Verify ood_stats.joblib
reloaded_ood = joblib.load(ood_stats_path)
assert reloaded_ood["metadata"]["features"] == CANONICAL_FEATURES
assert len(reloaded_ood["crop_stats"]) == 22
for crop in classes:
    assert crop in reloaded_ood["crop_stats"]
    assert reloaded_ood["crop_stats"][crop]["mean"].shape == (7,)
    assert reloaded_ood["crop_stats"][crop]["inv_cov"].shape == (7, 7)

# 2. Verify iforest.joblib
reloaded_if = joblib.load(iforest_path)
sample_test = np.array([[90.0, 42.0, 43.0, 21.0, 82.0, 6.5, 203.0]])
sample_sc = reloaded_if["scaler"].transform(sample_test)
sample_if_score = -reloaded_if["estimator"].score_samples(sample_sc)
assert sample_if_score.shape == (1,)

# 3. Verify thresholds.json
with open(thresholds_path, "r", encoding="utf-8") as f:
    reloaded_thresh = json.load(f)
assert "profiles" in reloaded_thresh
for prof in ["conservative", "balanced", "bold"]:
    assert prof in reloaded_thresh["profiles"]
    assert "ood_refuse_threshold" in reloaded_thresh["profiles"][prof]
    assert "ood_caution_threshold" in reloaded_thresh["profiles"][prof]

print("ALL SERIALIZED ARTIFACTS VERIFIED SUCCESSFULLY FROM DISK.")


ALL SERIALIZED ARTIFACTS VERIFIED SUCCESSFULLY FROM DISK.


## Summary of Findings & Next Steps

1. **Superiority of Per-Crop Mahalanobis Distance:**
   - On **Box-Uniform unknowns**, Mahalanobis achieves **0.9996 AUROC** with **0.0% FPR @ 95% TPR** (vs. 0.5000 AUROC for the naive 1D range check).
   - On **Shuffled-Column unknowns**, Mahalanobis achieves **0.9778 AUROC** (vs. 0.5987 for Isolation Forest), proving that tree splits struggle to catch broken correlation structures that covariance precision matrices easily detect.
2. **Leave-One-Crop-Out (LOCO) Robustness:**
   - Mean AUROC across all 22 crops is **0.9797 $\pm$ 0.0526** (evaluated strictly on validation rows, zero test leakage).
   - Distinct crops (`apple`, `banana`, `chickpea`, `watermelon`, `grapes`) achieve **1.0000 AUROC**.
   - Agronomically overlapping crops (`jute` with `rice`) achieve **0.7934 AUROC**, honestly documenting known confusable boundaries.
3. **Midpoint Traps:**
   - Intermediate points between confusable crops (e.g., rice and jute) score low Mahalanobis distance ($d^2 = 1.07$, 0th percentile), but midpoints between distant clusters (e.g., lentil and chickpea) score high distance ($d^2 = 22.67$, 100th percentile) and land outside known densities.
4. **Artifacts Ready:**
   - `backend/models/ood_stats.joblib`
   - `backend/models/iforest.joblib`
   - `backend/models/thresholds.json`
